# Results

In [1]:
import duckdb
import pandas as pd
import sys
import spacy
import os
sys.path.append('..')

from src.utils import make_corpus, preprocess_spacy
from src.semantic import build_semantic_index, semantic_search
from src.bm25 import bm25_search, bm25_tokenize

## Build Corpus and Preprocessing

In [2]:
# Read data and drop missing values
c2 = duckdb.connect()
data = c2.execute(f"SELECT * FROM read_parquet('../data/raw/merged.parquet')").df()
data.dropna(subset=['product_title'], inplace=True)

In [3]:
# Extract fields for retrieval
cols = ['product_title', 'main_category', 'store', 'title', 'text']

corpus = make_corpus(df=data, cols=cols, asin="asin")

In [4]:
# preprocess corpus and save it
os.makedirs('data/processed', exist_ok=True)

# if corpus is already processed and saved, pass to save time
corpus_path = "../data/processed/preprocessed_corpus.csv"
if os.path.exists(corpus_path):
    corpus = pd.read_csv(corpus_path)
else:
    nlp = spacy.load("en_core_web_md", disable=["parser", "ner"])
    corpus["text"] = [preprocess_spacy(text) for text in nlp.pipe(corpus["text"])]
    corpus.to_csv(corpus_path)

## Save Indices for BM25 and Embeddings

In [5]:
# BM25 index
from rank_bm25 import BM25Okapi


# save the BM25 index into pickle file
import pickle

pickle_path = "../data/processed/bm25.pkl"

if not os.path.exists(pickle_path):
    # tokenize corpus
    tokenized_products = [text.split() for text in corpus["text"]]
    bm25 = BM25Okapi(tokenized_products)
    
    # save to pickle
    with open(pickle_path, "wb") as f:
        pickle.dump(bm25, f)

# load it
with open(pickle_path, "rb") as f:
    bm25 = pickle.load(f)

In [6]:
# Semantic index 
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
semantic_index_path = '../data/processed/embedding.faiss'

if not os.path.exists(semantic_index_path):
    build_semantic_index(corpus, model, semantic_index_path)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Retrieve Results

In [7]:
queries = ["Wet wipes",
           "Bar Soap"]

In [8]:
for q in queries:
    print(f"QUERY: {q}\n")

    print("BM25 top results:")
    display(bm25_search(q, bm25, data, top_k=10))

    print("\nSemantic search top results:")
    display(semantic_search(q, semantic_index_path, model, data, top_k=10))

QUERY: Wet wipes

BM25 top results:


,product_title,text,rating,score
9003,Rolhei 75% Ethanol Wet Wipe - 2 Packs of 100 (...,that the wipes are thick and not thin.,5.0,15.452317
6634,Lens Wipes Pre-moistened Eye Glasses Cleaner W...,"I bought these based on the reviews, but they ...",1.0,12.032342
12808,"Pre-Moistened Lens Cleaning Wipes, Wet and Dry...",These were really wipes more for use in medica...,1.0,11.516642
6716,"Pre-Moistened Lens Cleaning Wipes, Wet and Dry...",What we liked most was that it does an excelle...,2.0,11.441593
15207,"Pre-Moistened Lens Cleaning Wipes, Wet and Dry...",Do not buy this item. The wipes are so small t...,1.0,10.531983
525,"Pre-Moistened Lens Cleaning Wipes, Wet and Dry...",The one positive feature of this Lens Cleaning...,2.0,10.241038
12775,EBPP Odor & Stain Eraser - Made in The USA - P...,Used within minutes of receiving and this stuf...,5.0,9.224869
863,Rinse Free Sponge Bath Wipes (30-pack) | Extra...,"I didn’t expect much from these, and I didn’t ...",3.0,8.964907
8375,"Mr Clean Magic Eraser Pads, 8 Count (Pack of 1)",Not sure what kind of aliens we stole this mat...,5.0,8.742500
12542,"Gonioa Baby Wipes Dispenser, Baby Wipes Case, ...",I bought this to put in my dining room so I ha...,5.0,8.710184



Semantic search top results:


,product_title,text,rating,score
799,"Travelon Hand Soap Toiletry Sheets, 50-Count",Did the job. Definitely remember - dry fingers...,5.0,0.978040
7009,Pampers Baby Fresh Water Baby Wipes 3X Pop-Top...,These are great wipes for on the go! I keep so...,5.0,0.970308
863,Rinse Free Sponge Bath Wipes (30-pack) | Extra...,"I didn’t expect much from these, and I didn’t ...",3.0,0.951311
17282,Pampers Baby Fresh Water Baby Wipes 3X Pop-Top...,These wipes are good for just about anything i...,5.0,0.946627
9003,Rolhei 75% Ethanol Wet Wipe - 2 Packs of 100 (...,that the wipes are thick and not thin.,5.0,0.932010
460,Pampers Baby Fresh Water Baby Wipes 3X Pop-Top...,Baby wipes sure have improved since I used the...,5.0,0.920217
6634,Lens Wipes Pre-moistened Eye Glasses Cleaner W...,"I bought these based on the reviews, but they ...",1.0,0.876085
2377,Rinse Free Sponge Bath Wipes (30-pack) | Extra...,I was very pleased with these rinse-free bath ...,4.0,0.851884
4990,Wet-it Skrubba New European Scrubby Non-Scratc...,"Love these scrubbers, colorful and work well, ...",5.0,0.849512
230,Rinse Free Sponge Bath Wipes (30-pack) | Extra...,When you can't get in the shower and want to f...,4.0,0.840174


QUERY: Bar Soap

BM25 top results:


,product_title,text,rating,score
16384,Zero Waste Dish Washing Soap Bar Set (Cinnamon...,Love it. No mess. Works as well as liquid dish...,5.0,12.696275
4172,Zero Waste Dish Washing Soap Bar Set (Cinnamon...,I'm trying to find products to replace all the...,4.0,12.650885
12804,Dealglad 10Pcs Double Layer Exfoliating Mesh S...,Must have with bars of soap ! You will love !,5.0,12.193924
4991,Dial Corp. 04303 Fels-Naptha Laundry Bar Soap ...,I use this along with other soaps as an inexpe...,5.0,12.046332
4371,Bubble Shack Hawaii Loofah Soap Trio Organza S...,"I like these loofah soaps. These, however, see...",4.0,11.755763
2883,Dial Corp. 04303 Fels-Naptha Laundry Bar Soap ...,Mom used this soap and I use it now too. This ...,5.0,11.754845
4935,"Travelon Hand Soap Toiletry Sheets, 50-Count","If you MUST save on space, consider these laun...",3.0,11.636635
2334,Dial Corp. 04303 Fels-Naptha Laundry Bar Soap ...,"A good laundry product, I keep coming back for...",5.0,10.928236
13141,"Travelon Hand Soap Toiletry Sheets, 50-Count",Although this had decent reviews when I resear...,3.0,10.721776
6378,"Travelon Hand Soap Toiletry Sheets, 50-Count",These were better than nothing but not that gr...,2.0,10.166126



Semantic search top results:


,product_title,text,rating,score
10602,Bubble Shack Hawaii Loofah Soap Trio Organza S...,These are great! They really help with polish...,5.0,0.965136
844,Beautywin Soft Silicone Bath Brush，Baby Shower...,Soap just falls right out it feel nice but ur ...,1.0,0.863133
2334,Dial Corp. 04303 Fels-Naptha Laundry Bar Soap ...,"A good laundry product, I keep coming back for...",5.0,0.859523
4172,Zero Waste Dish Washing Soap Bar Set (Cinnamon...,I'm trying to find products to replace all the...,4.0,0.850670
10720,Dial Corp. 04303 Fels-Naptha Laundry Bar Soap ...,Best product for the money. It gets out stains...,5.0,0.824575
4371,Bubble Shack Hawaii Loofah Soap Trio Organza S...,"I like these loofah soaps. These, however, see...",4.0,0.808832
4991,Dial Corp. 04303 Fels-Naptha Laundry Bar Soap ...,I use this along with other soaps as an inexpe...,5.0,0.807457
16384,Zero Waste Dish Washing Soap Bar Set (Cinnamon...,Love it. No mess. Works as well as liquid dish...,5.0,0.794553
12804,Dealglad 10Pcs Double Layer Exfoliating Mesh S...,Must have with bars of soap ! You will love !,5.0,0.786824
2883,Dial Corp. 04303 Fels-Naptha Laundry Bar Soap ...,Mom used this soap and I use it now too. This ...,5.0,0.769150
